<a href="https://colab.research.google.com/github/inyangelarissa/product_recommendation_system_Group16/blob/main/cli_app_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q opencv-python-headless scikit-image librosa soundfile joblib scikit-learn pandas numpy

In [7]:
import os

FILES_TO_CHECK = [
    "face_model.pkl", "face_scaler.pkl", "face_feature_columns.pkl", "face_class_names.pkl",
    "voice_model.pkl",
    "product_model.pkl", "merged_customer_dataset.csv",
]

for f in FILES_TO_CHECK:
    status = "FOUND" if os.path.exists(f) else "MISSING"
    print(f"[{status}] {f}")


[FOUND] face_model.pkl
[FOUND] face_scaler.pkl
[FOUND] face_feature_columns.pkl
[FOUND] face_class_names.pkl
[FOUND] voice_model.pkl
[FOUND] product_model.pkl
[FOUND] merged_customer_dataset.csv


In [8]:
FACE_MODEL_PATH = "face_model.pkl"
FACE_SCALER_PATH = "face_scaler.pkl"
FACE_FEATURE_COLS_PATH = "face_feature_columns.pkl"
FACE_CLASS_NAMES_PATH = "face_class_names.pkl"

VOICE_MODEL_PATH = "voice_model.pkl"

PRODUCT_MODEL_PATH = "product_model.pkl"
MERGED_DATA_PATH = "merged_customer_dataset.csv"

FACE_CONFIDENCE_THRESHOLD = 0.40
VOICE_CONFIDENCE_THRESHOLD = 0.40


In [9]:
import warnings
warnings.filterwarnings("ignore")

import cv2
import joblib
import numpy as np
import pandas as pd
from skimage.feature import hog


In [10]:
def extract_image_features(path: str) -> dict:
    img_bgr = cv2.imread(path)
    if img_bgr is None:
        raise ValueError(f"Could not read image: {path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    record = {}
    for ch in range(3):
        hist = cv2.calcHist([img_rgb], [ch], None, [8], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        for i, v in enumerate(hist):
            record[f"color_hist_{ch * 8 + i}"] = float(v)

    gray_hist = cv2.calcHist([gray], [0], None, [16], [0, 256])
    gray_hist = cv2.normalize(gray_hist, gray_hist).flatten()
    for i, v in enumerate(gray_hist):
        record[f"gray_hist_{i}"] = float(v)

    resized = cv2.resize(gray, (64, 64))
    hog_feats = hog(resized, orientations=9, pixels_per_cell=(16, 16),
                     cells_per_block=(1, 1), feature_vector=True)
    for i, v in enumerate(hog_feats):
        record[f"hog_{i}"] = float(v)

    return record


def extract_audio_features(path: str) -> dict:
    import librosa

    SR = 22050
    y, sr = librosa.load(path, sr=SR)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    rms = librosa.feature.rms(y=y)
    zcr = librosa.feature.zero_crossing_rate(y=y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

    feats = {}
    for i in range(mfcc.shape[0]):
        feats[f"mfcc{i+1}_mean"] = float(np.mean(mfcc[i]))
        feats[f"mfcc{i+1}_std"] = float(np.std(mfcc[i]))
    feats["spectral_rolloff_mean"] = float(np.mean(rolloff))
    feats["spectral_rolloff_std"] = float(np.std(rolloff))
    feats["rms_energy_mean"] = float(np.mean(rms))
    feats["rms_energy_std"] = float(np.std(rms))
    feats["zero_crossing_rate_mean"] = float(np.mean(zcr))
    feats["spectral_centroid_mean"] = float(np.mean(centroid))
    return feats

print("Feature extraction functions ready.")


Feature extraction functions ready.


In [11]:
def check_face(image_path: str):
    """Returns (decision, predicted_member, confidence)."""
    model = joblib.load(FACE_MODEL_PATH)
    scaler = joblib.load(FACE_SCALER_PATH)
    feature_cols = joblib.load(FACE_FEATURE_COLS_PATH)

    feats = extract_image_features(image_path)
    row = pd.DataFrame([feats]).reindex(columns=feature_cols, fill_value=0.0)
    x_scaled = scaler.transform(row.values)

    proba = model.predict_proba(x_scaled)[0]
    best_idx = int(np.argmax(proba))
    predicted_member = model.classes_[best_idx]
    confidence = float(proba[best_idx])

    decision = "recognized" if confidence >= FACE_CONFIDENCE_THRESHOLD else "denied"
    return decision, predicted_member, confidence

print("check_face() ready.")


check_face() ready.


In [12]:
def check_voice(audio_path: str, claimed_member: str):
    """Returns (decision, predicted_member, confidence)."""
    bundle = joblib.load(VOICE_MODEL_PATH)
    model = bundle["model"]
    scaler = bundle["scaler"]
    label_encoder = bundle["label_encoder"]
    feature_cols = bundle["feature_cols"]

    feats = extract_audio_features(audio_path)
    row = pd.DataFrame([feats]).reindex(columns=feature_cols, fill_value=0.0)
    x_scaled = scaler.transform(row.values)

    proba = model.predict_proba(x_scaled)[0]
    best_idx = int(np.argmax(proba))
    predicted_member = label_encoder.inverse_transform([best_idx])[0]
    confidence = float(proba[best_idx])

    if confidence < VOICE_CONFIDENCE_THRESHOLD:
        return "denied", predicted_member, confidence
    if predicted_member != claimed_member:
        return "denied", predicted_member, confidence
    return "approved", predicted_member, confidence

print("check_voice() ready.")


check_voice() ready.


In [13]:
def predict_product(customer_id: str):
    bundle = joblib.load(PRODUCT_MODEL_PATH)
    model = bundle["model"]
    scaler = bundle["scaler"]
    label_encoder = bundle["label_encoder"]
    feature_cols = bundle["feature_cols"]

    merged = pd.read_csv(MERGED_DATA_PATH)
    merged["customer_id"] = merged["customer_id"].astype(str)
    match = merged[merged["customer_id"] == str(customer_id)]
    if match.empty:
        raise ValueError(f"customer_id '{customer_id}' not found in {MERGED_DATA_PATH}")

    row = match.iloc[[0]].reindex(columns=feature_cols, fill_value=0.0)
    x_scaled = scaler.transform(row.values)

    proba = model.predict_proba(x_scaled)[0]
    best_idx = int(np.argmax(proba))
    predicted_product = label_encoder.inverse_transform([best_idx])[0]
    confidence = float(proba[best_idx])
    return predicted_product, confidence

print("predict_product() ready.")


predict_product() ready.


In [14]:
def run_transaction(face_path: str, voice_path: str, customer_id: str):
    print("=" * 60)
    print("STEP 1: Face check")
    print("=" * 60)
    face_decision, face_member, face_conf = check_face(face_path)
    print(f"Predicted identity: {face_member}  (confidence: {face_conf:.2f})")

    if face_decision == "denied":
        print(f"\nACCESS DENIED -- face not recognized with sufficient confidence "
              f"(threshold {FACE_CONFIDENCE_THRESHOLD}).")
        return

    print(f"Face recognized as '{face_member}'. Proceeding to voice check...\n")

    print("=" * 60)
    print("STEP 2: Voice check")
    print("=" * 60)
    voice_decision, voice_member, voice_conf = check_voice(voice_path, claimed_member=face_member)
    print(f"Predicted speaker: {voice_member}  (confidence: {voice_conf:.2f})")

    if voice_decision == "denied":
        if voice_member != face_member:
            print(f"\nACCESS DENIED -- voice sounds like '{voice_member}', "
                  f"not '{face_member}' (identity mismatch).")
        else:
            print(f"\nACCESS DENIED -- voice confidence too low "
                  f"(threshold {VOICE_CONFIDENCE_THRESHOLD}).")
        return

    print(f"Voice confirmed as '{face_member}'. Proceeding to product prediction...\n")

    print("=" * 60)
    print("STEP 3: Product prediction")
    print("=" * 60)
    try:
        product, prod_conf = predict_product(customer_id)
    except ValueError as e:
        print(f"\nERROR -- {e}")
        return

    print(f"ACCESS GRANTED for '{face_member}' (customer_id={customer_id})")
    print(f"Predicted product: {product}  (confidence: {prod_conf:.2f})")

print("run_transaction() ready.")


run_transaction() ready.


In [27]:
run_transaction(
    face_path="rachel_smiling.jpeg",
    voice_path="/content/yes_approve (1).ogg",
    customer_id="100",
)

STEP 1: Face check
Predicted identity: rachel  (confidence: 0.94)
Face recognized as 'rachel'. Proceeding to voice check...

STEP 2: Voice check
Predicted speaker: rachel  (confidence: 0.94)
Voice confirmed as 'rachel'. Proceeding to product prediction...

STEP 3: Product prediction
ACCESS GRANTED for 'rachel' (customer_id=100)
Predicted product: Books  (confidence: 0.70)


In [31]:
run_transaction(
    face_path="/content/larissa_smiling.jpeg",
    voice_path="/content/yes_approve (1).ogg",
    customer_id="100",
)

STEP 1: Face check
Predicted identity: larissa  (confidence: 0.88)
Face recognized as 'larissa'. Proceeding to voice check...

STEP 2: Voice check
Predicted speaker: rachel  (confidence: 0.94)

ACCESS DENIED -- voice sounds like 'rachel', not 'larissa' (identity mismatch).
